# Week 5: SQL practice

Remember to save a copy with `_mine` in the filename.

## Setup

Use the course `.venv` kernel and your `_mine.ipynb` copy. This notebook expects its working directory to be `lessons/week05_relational-data`; its `../../data` path then reaches the course data. In Colab, extract the complete course folder under `/content/compss-211a` first. If loading fails, inspect `DATA` and check that the CSV files are present there.

Run the four setup cells in order. Expect a row count and a unique-ID check, a small requests table, and a district lookup table. SQLite is included with Python; no database server or account is required.

A **query** is a request for data. Here, `sql("SELECT 2 + 3 AS total")` returns a one-row table with `total` equal to 5. Triple quotes let a query span several lines. Replace each `____` or `...` in an exercise before running it; these are deliberate blanks.

If SQL reports “no such table,” rerun the database setup. If a Python name is undefined, rerun its defining cell. The database copies the DataFrames at setup time; changing a DataFrame afterward does not automatically change its SQL table.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

# The course data folder is two levels up from this notebook.
DATA = Path("../../data")
if not DATA.exists():  # Colab: the course folder sits in /content
    DATA = Path("/content/compss-211a/data")

In [ ]:
# One row per San Francisco 311 request, August 3-9, 2026.
# IDs are read as text. district is a whole number that can be missing.
raw = pd.read_csv(DATA / "sf311_requests.csv", dtype={"request_id": "string", "district": "Int64"})

# Leave out the rows whose channel is Test.
requests = raw[raw["channel"] != "Test"]

print(len(raw), "rows in the file,", len(requests), "kept")
print("Every request_id appears once:", requests["request_id"].is_unique)
requests.head(3)

In [ ]:
# One row per supervisor district. Keep two columns and give them shorter names.
districts = pd.read_csv(DATA / "sf_supervisor_districts.csv")
districts = districts.rename(columns={"sup_dist": "district", "sup_dist_name": "district_name"})
districts = districts[["district", "district_name"]]
districts

In [ ]:
# New this week: copy both tables into a database.
# This one lives in memory and disappears when the notebook closes.
database = sqlite3.connect(":memory:")
requests.to_sql("requests", database, index=False)
districts.to_sql("districts", database, index=False)

# sql("...") sends a query to the database and returns the answer as a pandas table.
def sql(query):
    return pd.read_sql_query(query, database)

## 1. From pandas to SQL

**Try pandas first (3 minutes).** Find Phone requests. Show `request_id`, `category`, and `district`, sorted by request ID. Keep the first five rows.

Save your result as `in_pandas`. You can use `.loc`, `.sort_values()`, and `.head()`.

In [ ]:
# Filter the rows, choose the columns, sort, and keep the first five.
in_pandas = ...
in_pandas

**Now run the same question in SQL.** Compare the two results.

| SQL | What you just did in pandas |
| --- | --- |
| `SELECT request_id, category, district` | Choose columns |
| `FROM requests` | Use the `requests` table |
| `WHERE channel = 'Phone'` | Keep rows that match the condition |
| `ORDER BY request_id` | Sort with `.sort_values("request_id")` |
| `LIMIT 5` | Keep the first five with `.head(5)` |

In [ ]:
in_sql = sql("""
    SELECT request_id, category, district
    FROM requests
    WHERE channel = 'Phone'
    ORDER BY request_id
    LIMIT 5
""")

display(in_pandas, in_sql)

**Your turn in SQL.** Choose another channel from `requests["channel"].unique()`. Show `request_id`, `status`, and `hours_to_close` for its first ten requests, sorted by request ID. Fill the blanks.

In [ ]:
sql("""
    SELECT request_id, ____, ____
    FROM requests
    WHERE channel = '____'
    ORDER BY request_id
    LIMIT ____
""")

## 2. Count groups

In Week 4, you used `.groupby()` to summarize requests. Run this pandas count, then its SQL equivalent. Both return one row per reporting channel.

In [ ]:
channel_counts = (
    requests.groupby("channel", dropna=False)
    .size()
    .reset_index(name="requests")
    .sort_values(["requests", "channel"], ascending=[False, True])
)
channel_counts

`GROUP BY channel` forms the groups; `COUNT(*)` counts their rows, like `.size()`. `AS requests` names the count column. `DESC` sorts largest first.

SQL includes missing group labels; pandas needs `dropna=False` to keep them.

In [ ]:
sql("""
    SELECT channel, COUNT(*) AS requests
    FROM requests
    GROUP BY channel
    ORDER BY requests DESC, channel ASC
""")

**Your turn.** How many Phone requests came from each district? Complete the query and show the largest count first.

In [ ]:
sql("""
    SELECT ____, COUNT(*) AS requests
    FROM requests
    WHERE channel = 'Phone'
    GROUP BY ____
    ORDER BY requests ____, district ASC
""")

## 3. Find missing values

You used `.isna()` in Week 4. Here it counts requests with no recorded closure time.

In [ ]:
requests["hours_to_close"].isna().sum()

In SQL, `IS NULL` identifies missing values. `WHERE` keeps those rows, and `COUNT(*)` counts them. Compare the count with pandas.

In [ ]:
sql("""
    SELECT COUNT(*) AS missing_time
    FROM requests
    WHERE hours_to_close IS NULL
""")

**Your turn.** Count requests with an unknown district. Fill the blank, then check your answer with `requests["district"].isna().sum()`.

In [ ]:
sql("""
    SELECT COUNT(*) AS requests
    FROM requests
    WHERE district ____
""")

Counting all rows and counting recorded values are different:

- `COUNT(*)` counts every row, like `len(requests)`.
- `COUNT(hours_to_close)` skips missing values, like `requests["hours_to_close"].count()`.
- `AVG(hours_to_close)` averages recorded values, like `requests["hours_to_close"].mean()`.

In [ ]:
# Worked example: compare all rows with observed durations.
sql("""
    SELECT COUNT(*) AS requests,
           COUNT(hours_to_close) AS with_time,
           AVG(hours_to_close) AS mean_hours
    FROM requests
""")

## 4. Join two tables

The `requests` table has a `district` column. The `districts` table gives the name for each number. Match those numbers to add `district_name` to each request.

Keep every request, including those with no matching district. This is a **left join**: `requests` is the left table.

In pandas, `.merge()` performs the join. Run the cell and compare the first three requests before and after it.

In [ ]:
joined_pandas = (
    requests[["request_id", "category", "district"]]
    .merge(districts, on="district", how="left")
    .sort_values("request_id")
)

display(
    requests[["request_id", "category", "district"]].sort_values("request_id").head(3),
    joined_pandas.head(3),
)

**Now do the same join in SQL.** `LEFT JOIN` corresponds to `how="left"`; `ON` specifies the matching columns, like `on="district"`.

`r` is short for `requests`; `d` is short for `districts`. Fill both blanks, then compare with pandas and run the checks.

In [ ]:
joined = sql("""
    SELECT r.request_id, r.category, r.district, d.district_name
    FROM requests AS r
    LEFT JOIN districts AS d
        ON r.____ = d.____
    ORDER BY r.request_id
""")
display(joined_pandas.head(3), joined.head(3))

In [ ]:
print("Before:", len(requests), "After:", len(joined))
print("Unique request IDs:", joined["request_id"].nunique())
print("Unmatched requests:", joined["district_name"].isna().sum())
assert len(joined) == len(requests)
assert joined["request_id"].notna().all() and joined["request_id"].is_unique
assert set(joined["request_id"]) == set(requests["request_id"])

## 5. Write your own SQL

Start from an empty cell; earlier examples are available for reference.

1. Show the five categories with the most **Open** requests, with their counts.
2. Choose a channel. Count its requests by **district name**, keeping requests without a match. This needs both tables.

For each query, first choose the source table, then the rows, grouping, and output columns. Inspect a small result before adding the sort. If a join unexpectedly increases the row count, check whether the lookup key appears more than once.

In [ ]:
# Write query 1 here.

In [ ]:
# Write query 2 here.

## 6. Food access by county

20 minutes, on your own. A food-access nonprofit wants to know which California counties to look at first.

`tracts` has one row per census tract, a small area with a few thousand residents. `counties` has one row per county. `low_income_low_access` is 1 when USDA flags a tract as low-income and far from a store that accepts SNAP benefits. Otherwise it is 0. [USDA definitions](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/snap-authorized-retailer-access-map-reference-guide).

A count answers “how many tracts?”; a share answers “what fraction of that county's tracts?” For example, 2 flagged tracts out of 8 is a share of 0.25. These are tract-level measures, not the fraction of individual residents with low access.

In [ ]:
# Load the two new tables and add them to the database. County codes stay text.
tracts = pd.read_csv(DATA / "week05_food_access_tracts.csv", dtype={"tract_id": "string", "county_fips": "string"})
counties = pd.read_csv(DATA / "week05_ca_counties.csv", dtype={"county_fips": "string"})
tracts.to_sql("tracts", database, index=False, if_exists="replace")
counties.to_sql("counties", database, index=False, if_exists="replace")

display(tracts.head(3), counties.head(3))

In [ ]:
# 1. Count the flagged tracts for each county name. Largest count first.
#    You need both tables. Name the columns county_name and flagged.
by_count = ...

In [ ]:
# Check: the counts add up to every flagged tract in the state.
all_flagged = sql("SELECT COUNT(*) AS tracts FROM tracts WHERE low_income_low_access = 1")
assert by_count["flagged"].sum() == all_flagged.loc[0, "tracts"]
by_count.head(5)

In [ ]:
# The flag is 1 or 0, so its sum is the number of flagged tracts
# and its average is the share of tracts that are flagged.
# Worked example: the whole state.
sql("""
    SELECT COUNT(*) AS tracts,
           SUM(low_income_low_access) AS flagged,
           AVG(low_income_low_access) AS share
    FROM tracts
""")

In [ ]:
# 2. Will the county with the most flagged tracts also have the largest share?
#    My guess:
#
#    Now make the same three numbers for each county name. Largest share first.
#    Name the columns county_name, tracts, flagged, and share.
by_share = ...

In [ ]:
# Check: one row per county, and the same flagged tracts as before.
assert len(by_share) == len(counties)
assert by_share["flagged"].sum() == by_count["flagged"].sum()

# The two top fives. Which list should the nonprofit see? We will discuss.
display(by_count.head(5), by_share.head(5))

## Optional browser practice

[SQLShed's playground](https://sqlshed.com/playground/) lets you import a CSV and try SQL; its [introductory lessons](https://sqlshed.com/learn/) cover the basics. Try a small public or made-up CSV with column headers. After importing, check the displayed table name and column types, then use `SELECT * FROM your_table LIMIT 5`, replacing `your_table` with that name.

SQLShed uses DuckDB; this notebook uses SQLite. The basic selection, grouping, and join ideas transfer, but some functions and imported types differ. This is optional practice: keep course exercises, checks, and saved results in your local notebook.